In [1]:
# -*- coding: utf-8 -*-
"""
Prevalence-aware SHAP imbalance analysis.

目的：
1. 分析标签不平衡是否影响 SHAP 解释结果；
2. 比较 all-sample SHAP 与 positive-sample SHAP；
3. 通过 balanced resampling SHAP 分析 Top 特征稳定性；
4. 给出每个 odor descriptor 的 SHAP 解释可靠性等级；
5. 输出表格和论文图，用于回应审稿意见中 label imbalance / label noise 对 SHAP interpretation 的影响。

输入：
1. 特征文件：
   ./Malodors_Rule&FG&Morgan&StructKG_features.xlsx

2. 之前 SHAP 代码保存的缓存目录：
   ./shap_cache_and_plots_24labels/cache/

可选输入：
3. 之前不平衡分析得到的模型性能表：
   ./imbalance_label_uncertainty_analysis_cleanplots/tables/per_label_imbalance_and_prediction_metrics.csv

输出：
./shap_imbalance_effect_analysis/

主要输出：
1. tables/per_label_shap_imbalance_summary.csv / xlsx
2. tables/shap_top_features_all_positive_balanced__{label}.csv
3. tables/balanced_shap_recurrence__{label}.csv
4. tables/all_labels_top_feature_recurrence_long.csv
5. plots/lollipop_topk_jaccard_all_vs_positive.png
6. plots/lollipop_balanced_shap_stability.png
7. plots/prevalence_vs_shap_stability.png
8. plots/shap_interpretation_level_bar.png
9. plots/top_feature_comparison__{label}.png
"""

import os
import re
import json
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager as fm

import xgboost as xgb
import shap

try:
    from scipy.stats import spearmanr
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

warnings.filterwarnings("ignore")


# ============================================================
# 0. 配置
# ============================================================

RANDOM_SEED = 42

FEATURE_FILE = "./Malodors_Rule&FG&Morgan&StructKG_features.xlsx"

# 你之前 SHAP 代码保存的缓存目录
OLD_SHAP_OUT_DIR = "./shap_cache_and_plots_24labels"
OLD_CACHE_DIR = os.path.join(OLD_SHAP_OUT_DIR, "cache")

# 本次分析输出目录
OUT_DIR = "./shap_imbalance_effect_analysis"
TABLE_DIR = os.path.join(OUT_DIR, "tables")
PLOT_DIR = os.path.join(OUT_DIR, "plots")
CACHE_DIR = os.path.join(OUT_DIR, "cache")

for d in [OUT_DIR, TABLE_DIR, PLOT_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

# 如果旧 SHAP 缓存不存在，是否允许重新训练并计算 SHAP
ALLOW_RECOMPUTE_MISSING_SHAP = True

# SHAP 重新计算时是否保存缓存
SAVE_RECOMPUTED_SHAP = True

# SHAP 保存类型
SHAP_SAVE_DTYPE = np.float16

# 分析 Top-K
TOPK = 20

# balanced SHAP resampling 次数
N_BALANCED_REPEATS = 100

# 每个标签最多画几个 top feature comparison
TOPK_PLOT_FEATURES = 15

# 如果阳性样本过少，balanced SHAP 仍然会做，但解释等级会自动降低
MIN_POS_FOR_STABLE_SHAP = 50
MIN_POS_FOR_HIGH_CONFIDENCE = 100

# 图片设置
DPI = 800
SAVE_XLSX = True

# 是否输出每个标签的 all vs positive vs balanced Top 特征对比图
PLOT_EACH_LABEL_TOP_COMPARISON = True

# 可选：读取模型性能表，用于把 AUPRC / AUPRC lift 合并进 SHAP 稳定性表
PERFORMANCE_TABLE_CANDIDATES = [
    "./imbalance_label_uncertainty_analysis_cleanplots/tables/per_label_imbalance_and_prediction_metrics.csv",
    "./imbalance_label_uncertainty_analysis/tables/per_label_imbalance_and_prediction_metrics.csv",
]


# ============================================================
# 1. 标签与模型参数
# ============================================================

TARGET_LABELS_24 = [
    "alcoholic", "aldehydic", "almond", "aromatic", "burnt", "cabbage",
    "cheesy", "cherry", "chocolate", "ethereal", "fishy", "fruity",
    "garlic", "grassy", "green", "ketonic", "musty", "pungent",
    "sharp", "solvent", "sour", "sulfurous", "sweaty", "sweet"
]

LABELS_138 = TARGET_LABELS_24.copy()

BEST_PARAMS = {
    "n_estimators": 433,
    "max_depth": 7,
    "learning_rate": 0.0350057872293877,
    "subsample": 0.9947153135691092,
    "colsample_bytree": 0.7778835626400454,
    "min_child_weight": 1.0072775841844182,
    "reg_lambda": 3.4681854273849724,
    "reg_alpha": 6.955456414716767e-08,
    "gamma": 3.606069985094933
}

BASE_XGB_PARAMS_SINGLE = dict(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbosity=0,
)


# ============================================================
# 2. 表格保存与绘图样式
# ============================================================

def save_table(df, name):
    csv_path = os.path.join(TABLE_DIR, f"{name}.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print("[SAVE]", csv_path)

    if SAVE_XLSX:
        xlsx_path = os.path.join(TABLE_DIR, f"{name}.xlsx")
        df.to_excel(xlsx_path, index=False)
        print("[SAVE]", xlsx_path)


def setup_plot_style():
    available_fonts = {f.name for f in fm.fontManager.ttflist}

    if "Arial" in available_fonts:
        font_family = "Arial"
    else:
        font_family = "DejaVu Sans"
        print("[WARN] Arial font not found. Matplotlib will use DejaVu Sans instead.")

    mpl.rcParams.update({
        "font.family": font_family,
        "font.weight": "bold",
        "axes.labelweight": "bold",
        "axes.linewidth": 1.8,
        "xtick.major.width": 1.8,
        "ytick.major.width": 1.8,
        "xtick.major.size": 5,
        "ytick.major.size": 5,
        "axes.unicode_minus": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    })


def style_axis(ax):
    ax.tick_params(
        axis="both",
        labelsize=12,
        width=1.8,
        length=5
    )

    for tick in ax.get_xticklabels():
        tick.set_fontweight("bold")

    for tick in ax.get_yticklabels():
        tick.set_fontweight("bold")

    for spine in ax.spines.values():
        spine.set_linewidth(1.8)


def save_fig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=DPI, bbox_inches="tight")
    plt.close()
    print("[SAVE]", path)


def wrap_label(s, width=28):
    s = str(s)

    if len(s) <= width:
        return s

    return "\n".join(
        re.findall(".{1,%d}(?:\\s+|$)|.{%d}" % (width, width), s)
    ).strip()


def shorten_feature_name(s):
    s = str(s)

    replace_map = {
        "KG_Ancestor__GroupsContainingSulfur": "S-containing groups",
        "KG_Ancestor__GroupsContainingNitrogen": "N-containing groups",
        "KG_Ancestor__Amines": "Amines",
        "KG_FG__Ether": "Ether",
        "FG: Carboxylic acid(–COOH)": "Carboxylic acid (-COOH)",
        "FG: Carboxylic acid(-COOH)": "Carboxylic acid (-COOH)",
        "FG: [CX3](=O)[#6][#6]": "Ketone-like carbonyl",
        "FG: [CX3H1](=O)[#6]": "Aldehyde",
        "C(=O)[OH] && (NumAliphaticCarbons >= 10)": "C(=O)[OH] && Aliphatic C≥10",
        "C(=O)[OH] && (MolWt < 110)": "C(=O)[OH] && MW<110",
        "C(=O)O&&(TPSA>60)&&(LogP<1.0)": "C(=O)O && TPSA>60 && LogP<1.0",
    }

    if s in replace_map:
        return replace_map[s]

    s = s.replace("KG_Ancestor__", "")
    s = s.replace("KG_FG__", "")
    s = s.replace("KG_Element__", "")
    s = s.replace("FG: ", "")
    s = s.replace("GroupsContainingSulfur", "S-containing groups")
    s = s.replace("GroupsContainingNitrogen", "N-containing groups")
    s = s.replace("NumAliphaticCarbons", "Aliphatic C")
    s = s.replace("MolWt", "MW")
    s = s.replace(">=", "≥")
    s = s.replace("<=", "≤")
    s = s.replace("&&", " && ")

    return s


# ============================================================
# 3. 读取特征文件，构建 X / y
# ============================================================

def find_smiles_col(df):
    cand = [c for c in df.columns if isinstance(c, str) and "smiles" in c.lower()]

    if not cand:
        return None

    for p in ["Canonical SMILES", "Canonical_SMILES", "canonical_smiles", "SMILES", "smiles", "StdSMILES"]:
        for c in cand:
            if c.lower() == p.lower():
                return c

    return cand[0]


def is_numeric_or_convertible(series):
    if np.issubdtype(series.dtype, np.number) or series.dtype == bool:
        return True

    try:
        pd.to_numeric(series, errors="raise")
        return True
    except Exception:
        return False


def build_X_y(df):
    smiles_col = find_smiles_col(df)

    missing = [c for c in TARGET_LABELS_24 if c not in df.columns]
    if missing:
        raise ValueError(f"缺少 24 个目标标签列：{missing}")

    y24 = df[TARGET_LABELS_24].fillna(0).astype(int).values

    exclude = set([c for c in LABELS_138 if c in df.columns])

    if smiles_col is not None:
        exclude.add(smiles_col)

    feat_cols = [c for c in df.columns if c not in exclude]

    good_cols = []
    bad_cols = []

    for c in feat_cols:
        if is_numeric_or_convertible(df[c]):
            good_cols.append(c)
        else:
            bad_cols.append(c)

    if bad_cols:
        print(f"[WARN] Dropped non-numeric X columns ({len(bad_cols)}):")
        print(bad_cols[:20])

    X_df = df[good_cols].copy()

    for c in X_df.columns:
        if not (np.issubdtype(X_df[c].dtype, np.number) or X_df[c].dtype == bool):
            X_df[c] = pd.to_numeric(X_df[c], errors="coerce")

    X = X_df.fillna(0).astype(np.float32).values

    return X, y24, good_cols, smiles_col


# ============================================================
# 4. SHAP 加载 / 重新计算
# ============================================================

def train_one_label(X, y_bin):
    params = dict(BASE_XGB_PARAMS_SINGLE)
    params.update(BEST_PARAMS)

    clf = xgb.XGBClassifier(**params)
    clf.fit(X, y_bin)

    return clf


def extract_shap_values(shap_output):
    """
    兼容不同 shap / xgboost 输出。
    """
    if isinstance(shap_output, list):
        if len(shap_output) == 2:
            return np.asarray(shap_output[1], dtype=np.float32)
        return np.asarray(shap_output[0], dtype=np.float32)

    return np.asarray(shap_output, dtype=np.float32)


def load_cached_shap(label):
    """
    优先读取旧 SHAP 缓存。
    """
    old_path = os.path.join(OLD_CACHE_DIR, f"shap_full__{label}.npz")

    if os.path.exists(old_path):
        data = np.load(old_path)
        sv = data["shap_values"].astype(np.float32)
        return sv, old_path

    new_path = os.path.join(CACHE_DIR, f"shap_full__{label}.npz")

    if os.path.exists(new_path):
        data = np.load(new_path)
        sv = data["shap_values"].astype(np.float32)
        return sv, new_path

    return None, None


def load_or_compute_shap(label, X, y_bin):
    sv, path = load_cached_shap(label)

    if sv is not None:
        print(f"[LOAD SHAP] {label}: {path}")
        return sv

    if not ALLOW_RECOMPUTE_MISSING_SHAP:
        raise FileNotFoundError(
            f"找不到 {label} 的 SHAP 缓存。请先运行原 SHAP 代码，或设置 ALLOW_RECOMPUTE_MISSING_SHAP=True。"
        )

    print(f"[RECOMPUTE SHAP] {label}: cache not found, training model...")

    model = train_one_label(X, y_bin)
    explainer = shap.TreeExplainer(model)
    shap_output = explainer.shap_values(X)
    sv = extract_shap_values(shap_output)

    if SAVE_RECOMPUTED_SHAP:
        save_path = os.path.join(CACHE_DIR, f"shap_full__{label}.npz")
        np.savez_compressed(
            save_path,
            shap_values=sv.astype(SHAP_SAVE_DTYPE)
        )
        print("[SAVE]", save_path)

    return sv


# ============================================================
# 5. SHAP 重要性与稳定性指标
# ============================================================

def normalize_importance(v):
    v = np.asarray(v, dtype=float)
    s = np.nansum(v)

    if not np.isfinite(s) or s <= 0:
        return np.zeros_like(v, dtype=float)

    return v / s


def topk_indices(importance, k=20):
    importance = np.asarray(importance, dtype=float)
    return np.argsort(importance)[::-1][:k]


def jaccard_topk(idx_a, idx_b):
    a = set([int(x) for x in idx_a])
    b = set([int(x) for x in idx_b])

    if len(a | b) == 0:
        return np.nan

    return len(a & b) / len(a | b)


def rank_spearman(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    mask = np.isfinite(a) & np.isfinite(b)

    if np.sum(mask) < 3:
        return np.nan

    if SCIPY_AVAILABLE:
        rho, _ = spearmanr(a[mask], b[mask])
        return float(rho) if np.isfinite(rho) else np.nan

    # fallback
    ar = pd.Series(a[mask]).rank().values
    br = pd.Series(b[mask]).rank().values

    if np.std(ar) == 0 or np.std(br) == 0:
        return np.nan

    return float(np.corrcoef(ar, br)[0, 1])


def distribution_shift_l1(imp_a, imp_b):
    """
    两个 SHAP importance 分布的 L1 shift。
    0 表示完全一致；越大表示 all-sample 与 positive-sample 越不一致。
    """
    pa = normalize_importance(imp_a)
    pb = normalize_importance(imp_b)

    return float(np.sum(np.abs(pa - pb)) / 2.0)


def compute_mean_abs_shap(sv, indices=None):
    if indices is None:
        return np.mean(np.abs(sv), axis=0).astype(np.float32)

    if len(indices) == 0:
        return np.full(sv.shape[1], np.nan, dtype=np.float32)

    return np.mean(np.abs(sv[indices]), axis=0).astype(np.float32)


def pairwise_jaccard(top_sets):
    """
    top_sets: list of arrays
    """
    if len(top_sets) < 2:
        return np.nan

    vals = []

    for i in range(len(top_sets)):
        for j in range(i + 1, len(top_sets)):
            vals.append(jaccard_topk(top_sets[i], top_sets[j]))

    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]

    if len(vals) == 0:
        return np.nan

    return float(np.mean(vals))


def balanced_shap_resampling(
    sv,
    pos_idx,
    neg_idx,
    n_repeats=100,
    topk=20,
    random_seed=42
):
    """
    balanced SHAP sensitivity analysis.

    每次使用：
    - 所有 positive samples；
    - 随机抽取相同数量的 negative samples；
    计算 mean(|SHAP|) 和 Top-K features。

    目的：
    检查在不让阴性样本数量主导解释结果时，Top SHAP features 是否稳定。
    """
    rng = np.random.default_rng(random_seed)

    pos_idx = np.asarray(pos_idx, dtype=int)
    neg_idx = np.asarray(neg_idx, dtype=int)

    n_pos = len(pos_idx)
    n_neg = len(neg_idx)

    if n_pos == 0 or n_neg == 0:
        return None

    sample_neg_n = min(n_pos, n_neg)

    all_importances = []
    all_top_idx = []

    for r in range(n_repeats):
        sampled_neg = rng.choice(
            neg_idx,
            size=sample_neg_n,
            replace=False if n_neg >= sample_neg_n else True
        )

        idx = np.concatenate([pos_idx, sampled_neg])
        imp = compute_mean_abs_shap(sv, idx)

        all_importances.append(imp)
        all_top_idx.append(topk_indices(imp, topk))

    all_importances = np.vstack(all_importances)

    mean_importance = np.nanmean(all_importances, axis=0)
    std_importance = np.nanstd(all_importances, axis=0)

    recurrence = np.zeros(sv.shape[1], dtype=float)

    for top in all_top_idx:
        recurrence[top] += 1

    recurrence = recurrence / float(n_repeats)

    stability_jaccard = pairwise_jaccard(all_top_idx)

    return {
        "balanced_importance_mean": mean_importance.astype(np.float32),
        "balanced_importance_std": std_importance.astype(np.float32),
        "topk_recurrence": recurrence.astype(np.float32),
        "balanced_topk_stability_jaccard": stability_jaccard,
        "n_balanced_repeats": n_repeats,
        "sample_neg_n": sample_neg_n,
    }


def interpretation_level(row):
    """
    给每个 descriptor 一个 SHAP 解释可靠性等级。

    这个等级不是模型性能指标，而是告诉审稿人：
    哪些标签的 SHAP 解释可以较可靠地讨论，
    哪些只能作为探索性假设。
    """
    n_pos = row["n_pos"]
    prevalence = row["prevalence"]
    jaccard_all_pos = row["topk_jaccard_all_vs_positive"]
    balanced_stability = row["balanced_topk_stability_jaccard"]
    top1_recur = row["balanced_top1_recurrence"]
    auprc = row.get("AUPRC", np.nan)
    auprc_lift = row.get("AUPRC_lift_over_prevalence", np.nan)

    high_perf = True
    if np.isfinite(auprc_lift):
        high_perf = auprc_lift >= 5.0
    elif np.isfinite(auprc):
        high_perf = auprc >= 0.5

    if (
        n_pos >= MIN_POS_FOR_HIGH_CONFIDENCE
        and jaccard_all_pos >= 0.45
        and balanced_stability >= 0.45
        and top1_recur >= 0.70
        and high_perf
    ):
        return "high-confidence"

    if (
        n_pos >= MIN_POS_FOR_STABLE_SHAP
        and jaccard_all_pos >= 0.30
        and balanced_stability >= 0.30
        and top1_recur >= 0.50
    ):
        return "moderate-confidence"

    return "exploratory"


# ============================================================
# 6. 单标签 SHAP imbalance 分析
# ============================================================

def analyze_one_label_shap_imbalance(label, li, X, y24, feat_cols):
    print("\n" + "=" * 100)
    print(f"[SHAP IMBALANCE] {label} ({li + 1}/{len(TARGET_LABELS_24)})")
    print("=" * 100)

    y_bin = y24[:, li].astype(int)

    n_samples = len(y_bin)
    pos_idx = np.where(y_bin == 1)[0]
    neg_idx = np.where(y_bin == 0)[0]

    n_pos = int(len(pos_idx))
    n_neg = int(len(neg_idx))
    prevalence = n_pos / n_samples if n_samples > 0 else np.nan

    print(f"[INFO] n_pos={n_pos}, n_neg={n_neg}, prevalence={prevalence:.4f}")

    sv = load_or_compute_shap(label, X, y_bin)

    if sv.shape[0] != X.shape[0]:
        raise ValueError(
            f"{label} SHAP 样本数与 X 不一致：SHAP={sv.shape}, X={X.shape}"
        )

    if sv.shape[1] != X.shape[1]:
        raise ValueError(
            f"{label} SHAP 特征数与 X 不一致：SHAP={sv.shape}, X={X.shape}"
        )

    imp_all = compute_mean_abs_shap(sv)
    imp_pos = compute_mean_abs_shap(sv, pos_idx)
    imp_neg = compute_mean_abs_shap(sv, neg_idx)

    top_all = topk_indices(imp_all, TOPK)
    top_pos = topk_indices(imp_pos, TOPK)
    top_neg = topk_indices(imp_neg, TOPK)

    j_all_pos = jaccard_topk(top_all, top_pos)
    j_all_neg = jaccard_topk(top_all, top_neg)
    j_pos_neg = jaccard_topk(top_pos, top_neg)

    rho_all_pos = rank_spearman(imp_all, imp_pos)
    rho_all_neg = rank_spearman(imp_all, imp_neg)
    rho_pos_neg = rank_spearman(imp_pos, imp_neg)

    shift_all_pos = distribution_shift_l1(imp_all, imp_pos)
    shift_all_neg = distribution_shift_l1(imp_all, imp_neg)
    shift_pos_neg = distribution_shift_l1(imp_pos, imp_neg)

    balanced = balanced_shap_resampling(
        sv=sv,
        pos_idx=pos_idx,
        neg_idx=neg_idx,
        n_repeats=N_BALANCED_REPEATS,
        topk=TOPK,
        random_seed=RANDOM_SEED + li
    )

    if balanced is not None:
        imp_bal_mean = balanced["balanced_importance_mean"]
        imp_bal_std = balanced["balanced_importance_std"]
        recurrence = balanced["topk_recurrence"]
        bal_stability = balanced["balanced_topk_stability_jaccard"]
        sample_neg_n = balanced["sample_neg_n"]

        top_bal = topk_indices(imp_bal_mean, TOPK)

        j_all_bal = jaccard_topk(top_all, top_bal)
        j_pos_bal = jaccard_topk(top_pos, top_bal)

        rho_all_bal = rank_spearman(imp_all, imp_bal_mean)
        rho_pos_bal = rank_spearman(imp_pos, imp_bal_mean)

        shift_all_bal = distribution_shift_l1(imp_all, imp_bal_mean)
        shift_pos_bal = distribution_shift_l1(imp_pos, imp_bal_mean)

        top1_bal_idx = int(top_bal[0])
        top1_recur = float(recurrence[top1_bal_idx])

    else:
        imp_bal_mean = np.full_like(imp_all, np.nan)
        imp_bal_std = np.full_like(imp_all, np.nan)
        recurrence = np.full_like(imp_all, np.nan)

        bal_stability = np.nan
        sample_neg_n = np.nan

        top_bal = np.array([], dtype=int)

        j_all_bal = np.nan
        j_pos_bal = np.nan
        rho_all_bal = np.nan
        rho_pos_bal = np.nan
        shift_all_bal = np.nan
        shift_pos_bal = np.nan
        top1_recur = np.nan

    # ========================================================
    # 保存 Top 特征对比表
    # ========================================================

    union_idx = list(
        dict.fromkeys(
            list(top_all)
            + list(top_pos)
            + list(top_neg)
            + list(top_bal)
        )
    )

    top_rows = []

    for idx in union_idx:
        top_rows.append({
            "label": label,
            "feature_index": int(idx),
            "feature": feat_cols[idx],
            "feature_short": shorten_feature_name(feat_cols[idx]),
            "mean_abs_shap_all": float(imp_all[idx]),
            "mean_abs_shap_positive": float(imp_pos[idx]),
            "mean_abs_shap_negative": float(imp_neg[idx]),
            "balanced_mean_abs_shap": float(imp_bal_mean[idx]) if np.isfinite(imp_bal_mean[idx]) else np.nan,
            "balanced_std_abs_shap": float(imp_bal_std[idx]) if np.isfinite(imp_bal_std[idx]) else np.nan,
            "balanced_topk_recurrence": float(recurrence[idx]) if np.isfinite(recurrence[idx]) else np.nan,
            "rank_all": int(np.where(np.argsort(imp_all)[::-1] == idx)[0][0] + 1),
            "rank_positive": int(np.where(np.argsort(imp_pos)[::-1] == idx)[0][0] + 1),
            "rank_negative": int(np.where(np.argsort(imp_neg)[::-1] == idx)[0][0] + 1),
            "rank_balanced": int(np.where(np.argsort(imp_bal_mean)[::-1] == idx)[0][0] + 1) if np.any(np.isfinite(imp_bal_mean)) else np.nan,
            "in_topk_all": int(idx in top_all),
            "in_topk_positive": int(idx in top_pos),
            "in_topk_negative": int(idx in top_neg),
            "in_topk_balanced": int(idx in top_bal),
        })

    top_df = pd.DataFrame(top_rows)

    top_df = top_df.sort_values(
        ["in_topk_balanced", "balanced_topk_recurrence", "balanced_mean_abs_shap"],
        ascending=[False, False, False]
    )

    save_table(
        top_df,
        f"shap_top_features_all_positive_balanced__{label}"
    )

    # ========================================================
    # 保存 balanced recurrence 长表
    # ========================================================

    rec_idx = np.argsort(recurrence)[::-1][:max(50, TOPK)] if np.any(np.isfinite(recurrence)) else []

    rec_rows = []

    for idx in rec_idx:
        rec_rows.append({
            "label": label,
            "feature_index": int(idx),
            "feature": feat_cols[idx],
            "feature_short": shorten_feature_name(feat_cols[idx]),
            "balanced_topk_recurrence": float(recurrence[idx]),
            "balanced_mean_abs_shap": float(imp_bal_mean[idx]) if np.isfinite(imp_bal_mean[idx]) else np.nan,
            "balanced_std_abs_shap": float(imp_bal_std[idx]) if np.isfinite(imp_bal_std[idx]) else np.nan,
            "mean_abs_shap_all": float(imp_all[idx]),
            "mean_abs_shap_positive": float(imp_pos[idx]),
        })

    rec_df = pd.DataFrame(rec_rows)

    save_table(
        rec_df,
        f"balanced_shap_recurrence__{label}"
    )

    # ========================================================
    # 单标签 summary
    # ========================================================

    summary = {
        "label": label,
        "n_samples": int(n_samples),
        "n_pos": int(n_pos),
        "n_neg": int(n_neg),
        "prevalence": float(prevalence),
        "prevalence_percent": float(prevalence * 100),

        "topk": int(TOPK),

        "topk_jaccard_all_vs_positive": float(j_all_pos),
        "topk_jaccard_all_vs_negative": float(j_all_neg),
        "topk_jaccard_positive_vs_negative": float(j_pos_neg),
        "topk_jaccard_all_vs_balanced": float(j_all_bal) if np.isfinite(j_all_bal) else np.nan,
        "topk_jaccard_positive_vs_balanced": float(j_pos_bal) if np.isfinite(j_pos_bal) else np.nan,

        "rank_spearman_all_vs_positive": float(rho_all_pos) if np.isfinite(rho_all_pos) else np.nan,
        "rank_spearman_all_vs_negative": float(rho_all_neg) if np.isfinite(rho_all_neg) else np.nan,
        "rank_spearman_positive_vs_negative": float(rho_pos_neg) if np.isfinite(rho_pos_neg) else np.nan,
        "rank_spearman_all_vs_balanced": float(rho_all_bal) if np.isfinite(rho_all_bal) else np.nan,
        "rank_spearman_positive_vs_balanced": float(rho_pos_bal) if np.isfinite(rho_pos_bal) else np.nan,

        "shap_distribution_shift_l1_all_vs_positive": float(shift_all_pos),
        "shap_distribution_shift_l1_all_vs_negative": float(shift_all_neg),
        "shap_distribution_shift_l1_positive_vs_negative": float(shift_pos_neg),
        "shap_distribution_shift_l1_all_vs_balanced": float(shift_all_bal) if np.isfinite(shift_all_bal) else np.nan,
        "shap_distribution_shift_l1_positive_vs_balanced": float(shift_pos_bal) if np.isfinite(shift_pos_bal) else np.nan,

        "balanced_repeats": int(N_BALANCED_REPEATS),
        "balanced_negative_sample_size_per_repeat": int(sample_neg_n) if np.isfinite(sample_neg_n) else np.nan,
        "balanced_topk_stability_jaccard": float(bal_stability) if np.isfinite(bal_stability) else np.nan,
        "balanced_top1_recurrence": float(top1_recur) if np.isfinite(top1_recur) else np.nan,

        "top1_all": feat_cols[int(top_all[0])] if len(top_all) > 0 else "",
        "top1_positive": feat_cols[int(top_pos[0])] if len(top_pos) > 0 else "",
        "top1_balanced": feat_cols[int(top_bal[0])] if len(top_bal) > 0 else "",
    }

    return summary, top_df, rec_df


# ============================================================
# 7. 读取性能表并合并
# ============================================================

def load_performance_table():
    for p in PERFORMANCE_TABLE_CANDIDATES:
        if os.path.exists(p):
            print("[INFO] Loading performance table:", p)
            df = pd.read_csv(p)

            keep = [
                "label",
                "AUROC",
                "AUPRC",
                "AUPRC_baseline_prevalence",
                "AUPRC_lift_over_prevalence",
                "recall_at_0.5",
                "precision_at_0.5",
                "f1_at_0.5",
                "near_boundary_rate_0.4_0.6",
            ]

            keep = [c for c in keep if c in df.columns]

            return df[keep].copy()

    print("[WARN] Performance table not found. Only prevalence and SHAP stability will be reported.")
    return None


# ============================================================
# 8. 绘图
# ============================================================

def plot_lollipop(summary_df, metric, xlabel, out_name, sort_by="prevalence"):
    df = summary_df.copy()
    df = df[np.isfinite(df[metric])].copy()

    if df.empty:
        print(f"[WARN] Empty plot data: {metric}")
        return

    if sort_by == "prevalence":
        df = df.sort_values("prevalence_percent", ascending=True)
    elif sort_by == "metric":
        df = df.sort_values(metric, ascending=True)
    else:
        df = df.sort_values("label", ascending=True)

    y_pos = np.arange(len(df))

    fig_h = max(6.0, 0.36 * len(df))
    fig, ax = plt.subplots(figsize=(8.2, fig_h))

    ax.hlines(
        y=y_pos,
        xmin=0,
        xmax=df[metric].values,
        linewidth=2.0,
        alpha=0.55
    )

    ax.scatter(
        df[metric].values,
        y_pos,
        s=85,
        edgecolors="black",
        linewidths=0.7,
        zorder=3
    )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(
        df["label"].values,
        fontsize=11,
        fontweight="bold"
    )

    ax.set_xlabel(xlabel, fontsize=15, fontweight="bold")
    ax.set_ylabel("Odor descriptor", fontsize=15, fontweight="bold")
    ax.set_title("")
    ax.grid(axis="x", alpha=0.25)

    style_axis(ax)

    save_fig(os.path.join(PLOT_DIR, out_name))


def plot_prevalence_vs_shap_stability(summary_df):
    df = summary_df.copy()
    metric = "topk_jaccard_all_vs_positive"

    df = df[np.isfinite(df["prevalence_percent"]) & np.isfinite(df[metric])].copy()

    if df.empty:
        return

    fig, ax = plt.subplots(figsize=(7.6, 5.8))

    ax.scatter(
        df["prevalence_percent"],
        df[metric],
        s=90,
        alpha=0.9,
        edgecolors="black",
        linewidths=0.7
    )

    # 只标注关键标签，避免重叠
    key_labels = ["sulfurous", "fruity", "sweet", "green"]
    low_stability = df.sort_values(metric, ascending=True).head(3)["label"].tolist()

    key_labels = list(dict.fromkeys(key_labels + low_stability))

    for _, r in df.iterrows():
        if r["label"] in key_labels:
            ax.annotate(
                r["label"],
                xy=(r["prevalence_percent"], r[metric]),
                xytext=(6, 5),
                textcoords="offset points",
                fontsize=10,
                fontweight="bold"
            )

    ax.set_xlabel("Positive prevalence (%)", fontsize=15, fontweight="bold")
    ax.set_ylabel("Top-K Jaccard: all vs positive SHAP", fontsize=15, fontweight="bold")
    ax.set_title("")
    ax.grid(alpha=0.25)

    style_axis(ax)

    save_fig(os.path.join(PLOT_DIR, "prevalence_vs_shap_stability.png"))


def plot_interpretation_level_bar(summary_df):
    df = summary_df.copy()

    count_df = (
        df["shap_interpretation_level"]
        .value_counts()
        .reindex(["high-confidence", "moderate-confidence", "exploratory"])
        .fillna(0)
        .reset_index()
    )

    count_df.columns = ["level", "count"]

    fig, ax = plt.subplots(figsize=(6.5, 4.8))

    ax.bar(
        count_df["level"],
        count_df["count"]
    )

    ax.set_xlabel("SHAP interpretation level", fontsize=15, fontweight="bold")
    ax.set_ylabel("Number of descriptors", fontsize=15, fontweight="bold")
    ax.set_title("")
    ax.grid(axis="y", alpha=0.25)

    ax.set_xticklabels(
        count_df["level"],
        rotation=20,
        ha="right",
        fontsize=11,
        fontweight="bold"
    )

    style_axis(ax)

    save_fig(os.path.join(PLOT_DIR, "shap_interpretation_level_bar.png"))


def plot_top_feature_comparison(label, top_df):
    """
    每个标签一张图：
    比较 all / positive / balanced 三种 SHAP importance。
    """
    df = top_df.copy()

    if df.empty:
        return

    # 优先选择 balanced recurrence 高的特征
    df = df.sort_values(
        ["balanced_topk_recurrence", "balanced_mean_abs_shap"],
        ascending=False
    ).head(TOPK_PLOT_FEATURES)

    df = df.iloc[::-1].copy()

    y_pos = np.arange(len(df))

    fig, ax = plt.subplots(figsize=(9.2, max(5.8, 0.42 * len(df))))

    h = 0.25

    ax.barh(
        y_pos - h,
        df["mean_abs_shap_all"].values,
        height=h,
        label="All samples"
    )

    ax.barh(
        y_pos,
        df["mean_abs_shap_positive"].values,
        height=h,
        label="Positive samples"
    )

    ax.barh(
        y_pos + h,
        df["balanced_mean_abs_shap"].values,
        height=h,
        label="Balanced samples"
    )

    labels = [
        wrap_label(shorten_feature_name(x), width=34)
        for x in df["feature"].values
    ]

    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=9, fontweight="bold")

    ax.set_xlabel("mean(|SHAP|)", fontsize=15, fontweight="bold")
    ax.set_ylabel("Feature", fontsize=15, fontweight="bold")
    ax.set_title("")
    ax.grid(axis="x", alpha=0.25)

    ax.legend(frameon=False, fontsize=10)

    style_axis(ax)

    out_png = os.path.join(PLOT_DIR, f"top_feature_comparison__{label}.png")
    save_fig(out_png)


# ============================================================
# 9. 主程序
# ============================================================

def main():
    setup_plot_style()

    print("[INFO] Reading feature file:", FEATURE_FILE)
    df = pd.read_excel(FEATURE_FILE)

    X, y24, feat_cols, smiles_col = build_X_y(df)

    print("[INFO] X shape:", X.shape)
    print("[INFO] y shape:", y24.shape)
    print("[INFO] n_features:", len(feat_cols))

    if smiles_col is not None:
        print("[INFO] smiles_col:", smiles_col)

    # 保存元信息
    meta = {
        "feature_file": FEATURE_FILE,
        "old_shap_cache_dir": OLD_CACHE_DIR,
        "out_dir": OUT_DIR,
        "target_labels": TARGET_LABELS_24,
        "topk": TOPK,
        "n_balanced_repeats": N_BALANCED_REPEATS,
        "analysis": {
            "all_sample_shap": "mean(|SHAP|) over all samples",
            "positive_sample_shap": "mean(|SHAP|) over samples with y=1 for each descriptor",
            "negative_sample_shap": "mean(|SHAP|) over samples with y=0 for each descriptor",
            "balanced_sample_shap": "all positive samples plus an equal number of randomly sampled negative samples",
            "stability": "Top-K recurrence and pairwise Top-K Jaccard across balanced resampling runs"
        }
    }

    with open(os.path.join(OUT_DIR, "analysis_metadata.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    all_summary_rows = []
    all_recurrence_long = []

    for li, label in enumerate(TARGET_LABELS_24):
        summary, top_df, rec_df = analyze_one_label_shap_imbalance(
            label=label,
            li=li,
            X=X,
            y24=y24,
            feat_cols=feat_cols
        )

        all_summary_rows.append(summary)

        if not rec_df.empty:
            all_recurrence_long.append(rec_df)

        if PLOT_EACH_LABEL_TOP_COMPARISON:
            plot_top_feature_comparison(label, top_df)

    summary_df = pd.DataFrame(all_summary_rows)

    # 合并模型性能表
    perf_df = load_performance_table()

    if perf_df is not None:
        summary_df = summary_df.merge(perf_df, on="label", how="left")

    # 添加 SHAP 解释可靠性等级
    summary_df["shap_interpretation_level"] = summary_df.apply(
        interpretation_level,
        axis=1
    )

    # 排序：低 prevalence 在前，方便看低频标签
    summary_df = summary_df.sort_values(
        ["prevalence", "topk_jaccard_all_vs_positive"],
        ascending=[True, True]
    ).reset_index(drop=True)

    save_table(
        summary_df,
        "per_label_shap_imbalance_summary"
    )

    if all_recurrence_long:
        recurrence_all_df = pd.concat(all_recurrence_long, axis=0, ignore_index=True)

        save_table(
            recurrence_all_df,
            "all_labels_top_feature_recurrence_long"
        )

    # ========================================================
    # 绘图：用于论文 / 回复审稿人
    # ========================================================

    plot_lollipop(
        summary_df=summary_df,
        metric="prevalence_percent",
        xlabel="Positive prevalence (%)",
        out_name="lollipop_positive_prevalence.png",
        sort_by="prevalence"
    )

    plot_lollipop(
        summary_df=summary_df,
        metric="topk_jaccard_all_vs_positive",
        xlabel=f"Top-{TOPK} Jaccard: all-sample vs positive-sample SHAP",
        out_name="lollipop_topk_jaccard_all_vs_positive.png",
        sort_by="prevalence"
    )

    plot_lollipop(
        summary_df=summary_df,
        metric="balanced_topk_stability_jaccard",
        xlabel=f"Balanced SHAP Top-{TOPK} stability Jaccard",
        out_name="lollipop_balanced_shap_stability.png",
        sort_by="prevalence"
    )

    plot_lollipop(
        summary_df=summary_df,
        metric="shap_distribution_shift_l1_all_vs_positive",
        xlabel="SHAP importance distribution shift: all vs positive",
        out_name="lollipop_shap_distribution_shift_all_vs_positive.png",
        sort_by="prevalence"
    )

    plot_lollipop(
        summary_df=summary_df,
        metric="balanced_top1_recurrence",
        xlabel="Balanced SHAP top-1 feature recurrence",
        out_name="lollipop_balanced_top1_recurrence.png",
        sort_by="prevalence"
    )

    plot_prevalence_vs_shap_stability(summary_df)

    plot_interpretation_level_bar(summary_df)

    # ========================================================
    # 自动生成简短结论文本
    # ========================================================

    n_labels = len(summary_df)
    n_exploratory = int(np.sum(summary_df["shap_interpretation_level"] == "exploratory"))
    n_moderate = int(np.sum(summary_df["shap_interpretation_level"] == "moderate-confidence"))
    n_high = int(np.sum(summary_df["shap_interpretation_level"] == "high-confidence"))

    rare_df = summary_df[summary_df["prevalence"] < 0.05]
    common_df = summary_df[summary_df["prevalence"] >= 0.10]

    rare_jaccard = float(np.nanmean(rare_df["topk_jaccard_all_vs_positive"])) if not rare_df.empty else np.nan
    common_jaccard = float(np.nanmean(common_df["topk_jaccard_all_vs_positive"])) if not common_df.empty else np.nan

    rare_stability = float(np.nanmean(rare_df["balanced_topk_stability_jaccard"])) if not rare_df.empty else np.nan
    common_stability = float(np.nanmean(common_df["balanced_topk_stability_jaccard"])) if not common_df.empty else np.nan

    summary_text = f"""
Prevalence-aware SHAP imbalance analysis
========================================

Number of descriptors: {n_labels}

SHAP interpretation levels:
- high-confidence: {n_high}
- moderate-confidence: {n_moderate}
- exploratory: {n_exploratory}

Effect of descriptor imbalance:
- Mean Top-{TOPK} Jaccard between all-sample and positive-sample SHAP for rare labels (<5% prevalence): {rare_jaccard:.3f}
- Mean Top-{TOPK} Jaccard between all-sample and positive-sample SHAP for labels with prevalence >=10%: {common_jaccard:.3f}
- Mean balanced SHAP stability Jaccard for rare labels (<5% prevalence): {rare_stability:.3f}
- Mean balanced SHAP stability Jaccard for labels with prevalence >=10%: {common_stability:.3f}

Interpretation:
- Lower all-vs-positive SHAP overlap indicates that all-sample SHAP summaries may be influenced by the dominant negative class.
- Lower balanced SHAP stability indicates that Top SHAP features are more sensitive to the sampled negative background.
- Therefore, SHAP interpretations for rare or unstable descriptors should be considered exploratory.
- SHAP interpretations are more reliable for descriptors with sufficient positive support, higher all-vs-positive consistency, and stable balanced resampling results.
""".strip()

    summary_path = os.path.join(OUT_DIR, "shap_imbalance_summary_for_reviewer.txt")

    with open(summary_path, "w", encoding="utf-8") as f:
        f.write(summary_text)

    print("[SAVE]", summary_path)

    print("\n" + "=" * 100)
    print(summary_text)
    print("=" * 100)

    print("\n[DONE]")
    print("Tables:", TABLE_DIR)
    print("Plots:", PLOT_DIR)
    print("Summary:", summary_path)


if __name__ == "__main__":
    main()

[WARN] Arial font not found. Matplotlib will use DejaVu Sans instead.
[INFO] Reading feature file: ./Malodors_Rule&FG&Morgan&StructKG_features.xlsx
[INFO] X shape: (3756, 2595)
[INFO] y shape: (3756, 24)
[INFO] n_features: 2595
[INFO] smiles_col: Canonical_SMILES

[SHAP IMBALANCE] alcoholic (1/24)
[INFO] n_pos=105, n_neg=3651, prevalence=0.0280
[RECOMPUTE SHAP] alcoholic: cache not found, training model...
[SAVE] ./shap_imbalance_effect_analysis/cache/shap_full__alcoholic.npz
[SAVE] ./shap_imbalance_effect_analysis/tables/shap_top_features_all_positive_balanced__alcoholic.csv
[SAVE] ./shap_imbalance_effect_analysis/tables/shap_top_features_all_positive_balanced__alcoholic.xlsx
[SAVE] ./shap_imbalance_effect_analysis/tables/balanced_shap_recurrence__alcoholic.csv
[SAVE] ./shap_imbalance_effect_analysis/tables/balanced_shap_recurrence__alcoholic.xlsx
[SAVE] ./shap_imbalance_effect_analysis/plots/top_feature_comparison__alcoholic.png

[SHAP IMBALANCE] aldehydic (2/24)
[INFO] n_pos=143, n_

In [2]:
# -*- coding: utf-8 -*-
"""
Redraw prevalence vs SHAP stability plot without label overlap.

输入：
    ./shap_imbalance_effect_analysis/tables/per_label_shap_imbalance_summary.csv

输出：
    ./shap_imbalance_effect_analysis/redraw_plots/prevalence_vs_shap_stability_clean.png
    ./shap_imbalance_effect_analysis/redraw_plots/prevalence_vs_shap_stability_clean_mapping.csv
    ./shap_imbalance_effect_analysis/redraw_plots/prevalence_vs_shap_stability_clean_mapping.xlsx

功能：
1. 读取之前保存的 SHAP imbalance summary；
2. 绘制 Positive prevalence (%) vs Top-K Jaccard: all vs positive SHAP；
3. 避免标签重叠；
4. 重复 / 接近重复的点不再全部标注，而是整理到图右侧说明框和映射表中。
"""

import os
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager as fm


# ============================================================
# 0. 路径配置
# ============================================================

IN_CSV = "./shap_imbalance_effect_analysis/tables/per_label_shap_imbalance_summary.csv"

OUT_DIR = "./shap_imbalance_effect_analysis/redraw_plots"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_PNG = os.path.join(
    OUT_DIR,
    "prevalence_vs_shap_stability_clean.png"
)

OUT_MAPPING_CSV = os.path.join(
    OUT_DIR,
    "prevalence_vs_shap_stability_clean_mapping.csv"
)

OUT_MAPPING_XLSX = os.path.join(
    OUT_DIR,
    "prevalence_vs_shap_stability_clean_mapping.xlsx"
)

DPI = 800


# ============================================================
# 1. 绘图参数
# ============================================================

X_COL = "prevalence_percent"
Y_COL = "topk_jaccard_all_vs_positive"

X_LABEL = "Positive prevalence (%)"
Y_LABEL = "Top-K Jaccard: all vs positive SHAP"

# Jaccard 分组精度
# 0.01 表示把 0.6667 归为 0.67，把 0.739 归为 0.74
JACCARD_ROUND_DECIMALS = 2

# 只有这些关键标签会直接标在图上
KEY_LABELS = [
    "sulfurous",
    "fruity",
    "green",
    "sweet",
]

# 是否额外标注最高 prevalence 和最低 Jaccard 的点
ANNOTATE_MAX_PREVALENCE = True
ANNOTATE_MIN_JACCARD = True

# 如果某个 Jaccard 分组中的点数 >= 2，则不逐个标注，放进右侧说明框
MIN_GROUP_SIZE_FOR_SIDE_NOTE = 2

# 右侧说明框最多展示多少个分组
MAX_SIDE_NOTE_GROUPS = 8

# 每个分组最多在说明框中展示多少个标签，过多则省略
MAX_LABELS_PER_GROUP_IN_NOTE = 6

# 点大小
POINT_SIZE = 95

# 图尺寸
FIGSIZE = (9.8, 6.8)


# ============================================================
# 2. 样式函数
# ============================================================

def setup_plot_style():
    available_fonts = {f.name for f in fm.fontManager.ttflist}

    if "Arial" in available_fonts:
        font_family = "Arial"
    else:
        font_family = "DejaVu Sans"
        print("[WARN] Arial font not found. Matplotlib will use DejaVu Sans instead.")

    mpl.rcParams.update({
        "font.family": font_family,
        "font.weight": "bold",
        "axes.labelweight": "bold",
        "axes.linewidth": 2.0,
        "xtick.major.width": 2.0,
        "ytick.major.width": 2.0,
        "xtick.major.size": 6,
        "ytick.major.size": 6,
        "axes.unicode_minus": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    })


def style_axis(ax):
    ax.tick_params(
        axis="both",
        labelsize=14,
        width=2.0,
        length=6
    )

    for tick in ax.get_xticklabels():
        tick.set_fontweight("bold")

    for tick in ax.get_yticklabels():
        tick.set_fontweight("bold")

    for spine in ax.spines.values():
        spine.set_linewidth(2.0)


def wrap_text(s, width=45):
    return "\n".join(
        textwrap.wrap(
            str(s),
            width=width,
            break_long_words=False,
            break_on_hyphens=False
        )
    )


def format_label_list(labels, max_n=6):
    labels = list(labels)

    if len(labels) <= max_n:
        return ", ".join(labels)

    shown = ", ".join(labels[:max_n])
    return f"{shown}, ..."


# ============================================================
# 3. 数据读取与分组
# ============================================================

def load_summary_data():
    if not os.path.exists(IN_CSV):
        raise FileNotFoundError(
            f"找不到输入文件：{IN_CSV}\n"
            f"请确认你已经运行过 SHAP imbalance analysis 代码。"
        )

    df = pd.read_csv(IN_CSV)

    required_cols = ["label", X_COL, Y_COL]
    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        raise ValueError(
            f"输入表缺少必要列：{missing}\n"
            f"当前列名为：{df.columns.tolist()}"
        )

    df = df.copy()
    df = df[np.isfinite(df[X_COL]) & np.isfinite(df[Y_COL])].copy()

    df["label"] = df["label"].astype(str)

    # 按 Jaccard 近似值分组
    df["jaccard_group"] = df[Y_COL].round(JACCARD_ROUND_DECIMALS)

    return df


def build_group_mapping(df):
    rows = []

    grouped = df.groupby("jaccard_group", sort=False)

    for group_value, g in grouped:
        g_sorted = g.sort_values(X_COL, ascending=True)

        rows.append({
            "jaccard_group": group_value,
            "n_descriptors": len(g_sorted),
            "labels": "; ".join(g_sorted["label"].tolist()),
            "prevalence_min": g_sorted[X_COL].min(),
            "prevalence_max": g_sorted[X_COL].max(),
            "jaccard_mean": g_sorted[Y_COL].mean(),
        })

    mapping_df = pd.DataFrame(rows)
    mapping_df = mapping_df.sort_values(
        ["jaccard_group", "prevalence_min"],
        ascending=[False, True]
    ).reset_index(drop=True)

    return mapping_df


def save_mapping(mapping_df):
    mapping_df.to_csv(OUT_MAPPING_CSV, index=False, encoding="utf-8-sig")
    mapping_df.to_excel(OUT_MAPPING_XLSX, index=False)

    print("[SAVE]", OUT_MAPPING_CSV)
    print("[SAVE]", OUT_MAPPING_XLSX)


# ============================================================
# 4. 绘图函数
# ============================================================

def plot_clean_scatter(df, mapping_df):
    fig, ax = plt.subplots(figsize=FIGSIZE)

    # --------------------------------------------------------
    # 4.1 按 Jaccard 分组画点
    # --------------------------------------------------------
    unique_groups = sorted(df["jaccard_group"].unique())

    cmap = plt.cm.tab20
    group_to_color = {
        g: cmap(i % 20)
        for i, g in enumerate(unique_groups)
    }

    for group_value in unique_groups:
        g = df[df["jaccard_group"] == group_value]

        ax.scatter(
            g[X_COL],
            g[Y_COL],
            s=POINT_SIZE,
            color=group_to_color[group_value],
            edgecolors="black",
            linewidths=0.8,
            alpha=0.92,
            label=f"{group_value:.2f}"
        )

    # --------------------------------------------------------
    # 4.2 选择少数关键点进行图内标注
    # --------------------------------------------------------
    labels_to_annotate = set(KEY_LABELS)

    if ANNOTATE_MAX_PREVALENCE:
        idx_max_prev = df[X_COL].idxmax()
        labels_to_annotate.add(df.loc[idx_max_prev, "label"])

    if ANNOTATE_MIN_JACCARD:
        idx_min_j = df[Y_COL].idxmin()
        labels_to_annotate.add(df.loc[idx_min_j, "label"])

    # 同一组点数较多时，除关键点外不标注
    group_size_dict = df.groupby("jaccard_group")["label"].count().to_dict()

    for _, r in df.iterrows():
        lab = r["label"]
        group_value = r["jaccard_group"]
        group_size = group_size_dict.get(group_value, 1)

        if lab not in labels_to_annotate:
            continue

        # 对于密集区也只标注关键点
        x = r[X_COL]
        y = r[Y_COL]

        # 根据位置给不同偏移，减少局部重叠
        if lab.lower() == "fruity":
            xytext = (8, 0)
            ha = "left"
            va = "center"
        elif lab.lower() in ["green", "sweet"]:
            xytext = (8, 6)
            ha = "left"
            va = "bottom"
        elif lab.lower() == "sulfurous":
            xytext = (8, 6)
            ha = "left"
            va = "bottom"
        else:
            xytext = (6, 5)
            ha = "left"
            va = "bottom"

        ax.annotate(
            lab,
            xy=(x, y),
            xytext=xytext,
            textcoords="offset points",
            fontsize=11,
            fontweight="bold",
            ha=ha,
            va=va
        )

    # --------------------------------------------------------
    # 4.3 右侧说明框：把重复/接近重复的点改成分组说明
    # --------------------------------------------------------
    repeated_groups = mapping_df[
        mapping_df["n_descriptors"] >= MIN_GROUP_SIZE_FOR_SIDE_NOTE
    ].copy()

    repeated_groups = repeated_groups.sort_values(
        ["jaccard_group", "n_descriptors"],
        ascending=[False, False]
    ).head(MAX_SIDE_NOTE_GROUPS)

    note_lines = []

    if not repeated_groups.empty:
        note_lines.append("Grouped descriptors")
        note_lines.append("")

        for _, r in repeated_groups.iterrows():
            labels = str(r["labels"]).split("; ")
            label_text = format_label_list(
                labels,
                max_n=MAX_LABELS_PER_GROUP_IN_NOTE
            )

            line = f"J={r['jaccard_group']:.2f}: {label_text}"
            note_lines.append(line)

    note_text = "\n".join(note_lines)

    if note_text:
        ax.text(
            1.02,
            0.98,
            wrap_text(note_text, width=38),
            transform=ax.transAxes,
            fontsize=9,
            fontweight="bold",
            va="top",
            ha="left",
            bbox=dict(
                boxstyle="round,pad=0.45",
                facecolor="white",
                edgecolor="black",
                linewidth=1.2,
                alpha=0.95
            )
        )

    # --------------------------------------------------------
    # 4.4 坐标轴与图例
    # --------------------------------------------------------
    ax.set_xlabel(
        X_LABEL,
        fontsize=18,
        fontweight="bold"
    )

    ax.set_ylabel(
        Y_LABEL,
        fontsize=18,
        fontweight="bold"
    )

    ax.set_title("")
    ax.grid(alpha=0.25)

    style_axis(ax)

    # 仅保留 Jaccard group 图例，放在图内左上角
    # 如果你觉得这个图例多余，可以把下面整个 legend 注释掉
    legend = ax.legend(
        title="Rounded Jaccard",
        frameon=False,
        fontsize=8,
        title_fontsize=9,
        loc="lower right",
        ncol=1
    )

    for text in legend.get_texts():
        text.set_fontweight("bold")

    if legend.get_title() is not None:
        legend.get_title().set_fontweight("bold")

    # --------------------------------------------------------
    # 4.5 调整边界
    # --------------------------------------------------------
    x_min = max(0, df[X_COL].min() - 2)
    x_max = df[X_COL].max() + 4

    y_min = max(0, df[Y_COL].min() - 0.04)
    y_max = min(1.05, df[Y_COL].max() + 0.04)

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    # 给右侧说明框留空间
    plt.subplots_adjust(right=0.72)

    plt.savefig(
        OUT_PNG,
        dpi=DPI,
        bbox_inches="tight"
    )

    plt.close()

    print("[SAVE]", OUT_PNG)


# ============================================================
# 5. 主程序
# ============================================================

def main():
    setup_plot_style()

    df = load_summary_data()

    mapping_df = build_group_mapping(df)

    save_mapping(mapping_df)

    print("\n[INFO] Jaccard group mapping:")
    print(mapping_df[["jaccard_group", "n_descriptors", "labels"]])

    plot_clean_scatter(df, mapping_df)

    print("\n[DONE]")
    print("Output figure:", OUT_PNG)
    print("Mapping table:", OUT_MAPPING_XLSX)


if __name__ == "__main__":
    main()

[WARN] Arial font not found. Matplotlib will use DejaVu Sans instead.
[SAVE] ./shap_imbalance_effect_analysis/redraw_plots/prevalence_vs_shap_stability_clean_mapping.csv
[SAVE] ./shap_imbalance_effect_analysis/redraw_plots/prevalence_vs_shap_stability_clean_mapping.xlsx

[INFO] Jaccard group mapping:
   jaccard_group  n_descriptors  \
0           1.00              1   
1           0.90              4   
2           0.82              6   
3           0.74              8   
4           0.67              5   

                                              labels  
0                                            ketonic  
1                sweaty; cabbage; alcoholic; pungent  
2       aromatic; sharp; musty; sweet; green; fruity  
3  grassy; solvent; fishy; garlic; burnt; cheesy;...  
4         chocolate; sour; cherry; almond; aldehydic  
[SAVE] ./shap_imbalance_effect_analysis/redraw_plots/prevalence_vs_shap_stability_clean.png

[DONE]
Output figure: ./shap_imbalance_effect_analysis/redraw_pl

In [3]:
# -*- coding: utf-8 -*-
"""
Save plotting data for:
Positive prevalence (%) vs Top-K Jaccard: all vs positive SHAP

输入：
    ./shap_imbalance_effect_analysis/tables/per_label_shap_imbalance_summary.csv

输出：
    1. prevalence_vs_shap_stability_plot_data.csv / xlsx
       每个 odor descriptor 一行，用于自己重新绘图。

    2. prevalence_vs_shap_stability_group_mapping.csv / xlsx
       按近似 Jaccard 分组后的标签列表，用于处理重复点或接近重复点。
"""

import os
import numpy as np
import pandas as pd


# ============================================================
# 0. 路径配置
# ============================================================

IN_CSV = "./shap_imbalance_effect_analysis/tables/per_label_shap_imbalance_summary.csv"

OUT_DIR = "./shap_imbalance_effect_analysis/redraw_plot_data"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_PLOT_DATA_CSV = os.path.join(
    OUT_DIR,
    "prevalence_vs_shap_stability_plot_data.csv"
)

OUT_PLOT_DATA_XLSX = os.path.join(
    OUT_DIR,
    "prevalence_vs_shap_stability_plot_data.xlsx"
)

OUT_GROUP_MAPPING_CSV = os.path.join(
    OUT_DIR,
    "prevalence_vs_shap_stability_group_mapping.csv"
)

OUT_GROUP_MAPPING_XLSX = os.path.join(
    OUT_DIR,
    "prevalence_vs_shap_stability_group_mapping.xlsx"
)


# ============================================================
# 1. 需要保存的绘图变量
# ============================================================

X_COL = "prevalence_percent"
Y_COL = "topk_jaccard_all_vs_positive"

# 近似重复点分组精度
# 例如 0.739 会归为 0.74，0.6667 会归为 0.67
JACCARD_ROUND_DECIMALS = 2

# 关键标签，可用于你后续选择性标注
KEY_LABELS = [
    "sulfurous",
    "fruity",
    "green",
    "sweet",
]


# ============================================================
# 2. 主程序
# ============================================================

def main():
    if not os.path.exists(IN_CSV):
        raise FileNotFoundError(
            f"找不到输入文件：{IN_CSV}\n"
            f"请确认已经运行过 SHAP imbalance analysis 代码。"
        )

    df = pd.read_csv(IN_CSV)

    required_cols = [
        "label",
        X_COL,
        Y_COL,
    ]

    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        raise ValueError(
            f"输入表缺少必要列：{missing}\n"
            f"当前列名为：{df.columns.tolist()}"
        )

    # ========================================================
    # 2.1 整理逐标签绘图数据
    # ========================================================

    keep_cols = [
        "label",
        "n_samples",
        "n_pos",
        "n_neg",
        "prevalence",
        "prevalence_percent",
        "topk_jaccard_all_vs_positive",
        "topk_jaccard_all_vs_negative",
        "topk_jaccard_positive_vs_negative",
        "topk_jaccard_all_vs_balanced",
        "topk_jaccard_positive_vs_balanced",
        "rank_spearman_all_vs_positive",
        "shap_distribution_shift_l1_all_vs_positive",
        "balanced_topk_stability_jaccard",
        "balanced_top1_recurrence",
        "shap_interpretation_level",
        "AUPRC",
        "AUPRC_lift_over_prevalence",
        "recall_at_0.5",
    ]

    keep_cols = [c for c in keep_cols if c in df.columns]

    plot_df = df[keep_cols].copy()

    plot_df = plot_df[
        np.isfinite(plot_df[X_COL])
        & np.isfinite(plot_df[Y_COL])
    ].copy()

    plot_df["label"] = plot_df["label"].astype(str)

    # x / y 显式命名，方便你后续自己绘图
    plot_df["plot_x_positive_prevalence_percent"] = plot_df[X_COL]
    plot_df["plot_y_topk_jaccard_all_vs_positive"] = plot_df[Y_COL]

    # 近似重复点分组
    plot_df["jaccard_group_round2"] = plot_df[Y_COL].round(
        JACCARD_ROUND_DECIMALS
    )

    # 是否关键标签
    plot_df["is_key_label"] = plot_df["label"].str.lower().isin(
        [x.lower() for x in KEY_LABELS]
    )

    # 是否最高 prevalence
    max_prev = plot_df[X_COL].max()
    plot_df["is_max_prevalence"] = plot_df[X_COL] == max_prev

    # 是否最低 Jaccard
    min_jaccard = plot_df[Y_COL].min()
    plot_df["is_min_jaccard"] = plot_df[Y_COL] == min_jaccard

    # 是否建议在图中标注
    plot_df["suggest_annotate"] = (
        plot_df["is_key_label"]
        | plot_df["is_max_prevalence"]
        | plot_df["is_min_jaccard"]
    )

    # 按 prevalence 排序，和原图 y 方向顺序一致
    plot_df = plot_df.sort_values(
        ["prevalence_percent", "topk_jaccard_all_vs_positive"],
        ascending=[False, False]
    ).reset_index(drop=True)

    plot_df.to_csv(
        OUT_PLOT_DATA_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    plot_df.to_excel(
        OUT_PLOT_DATA_XLSX,
        index=False
    )

    print("[SAVE]", OUT_PLOT_DATA_CSV)
    print("[SAVE]", OUT_PLOT_DATA_XLSX)

    # ========================================================
    # 2.2 保存重复 / 接近重复点分组表
    # ========================================================

    group_rows = []

    for group_value, g in plot_df.groupby("jaccard_group_round2", sort=False):
        g = g.sort_values(
            "prevalence_percent",
            ascending=False
        )

        labels = g["label"].tolist()

        group_rows.append({
            "jaccard_group_round2": group_value,
            "n_descriptors": len(g),
            "labels": "; ".join(labels),
            "prevalence_min": g["prevalence_percent"].min(),
            "prevalence_max": g["prevalence_percent"].max(),
            "jaccard_min": g["topk_jaccard_all_vs_positive"].min(),
            "jaccard_max": g["topk_jaccard_all_vs_positive"].max(),
            "jaccard_mean": g["topk_jaccard_all_vs_positive"].mean(),
            "contains_key_label": any(
                str(x).lower() in [k.lower() for k in KEY_LABELS]
                for x in labels
            ),
        })

    mapping_df = pd.DataFrame(group_rows)

    mapping_df = mapping_df.sort_values(
        ["jaccard_group_round2", "n_descriptors"],
        ascending=[False, False]
    ).reset_index(drop=True)

    mapping_df.to_csv(
        OUT_GROUP_MAPPING_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    mapping_df.to_excel(
        OUT_GROUP_MAPPING_XLSX,
        index=False
    )

    print("[SAVE]", OUT_GROUP_MAPPING_CSV)
    print("[SAVE]", OUT_GROUP_MAPPING_XLSX)

    # ========================================================
    # 2.3 打印核心信息
    # ========================================================

    print("\n" + "=" * 100)
    print("绘图数据预览")
    print("=" * 100)

    print(
        plot_df[
            [
                "label",
                "plot_x_positive_prevalence_percent",
                "plot_y_topk_jaccard_all_vs_positive",
                "jaccard_group_round2",
                "suggest_annotate",
            ]
        ].head(30)
    )

    print("\n" + "=" * 100)
    print("重复 / 接近重复 Jaccard 分组")
    print("=" * 100)

    print(
        mapping_df[
            [
                "jaccard_group_round2",
                "n_descriptors",
                "labels",
            ]
        ]
    )

    print("\n[DONE]")
    print("Plot data:", OUT_PLOT_DATA_XLSX)
    print("Group mapping:", OUT_GROUP_MAPPING_XLSX)


if __name__ == "__main__":
    main()

[SAVE] ./shap_imbalance_effect_analysis/redraw_plot_data/prevalence_vs_shap_stability_plot_data.csv
[SAVE] ./shap_imbalance_effect_analysis/redraw_plot_data/prevalence_vs_shap_stability_plot_data.xlsx
[SAVE] ./shap_imbalance_effect_analysis/redraw_plot_data/prevalence_vs_shap_stability_group_mapping.csv
[SAVE] ./shap_imbalance_effect_analysis/redraw_plot_data/prevalence_vs_shap_stability_group_mapping.xlsx

绘图数据预览
        label  plot_x_positive_prevalence_percent  \
0      fruity                           50.532481   
1       green                           38.391906   
2       sweet                           37.912673   
3   sulfurous                           10.596379   
4    ethereal                            7.641108   
5       musty                            5.298190   
6     pungent                            5.244941   
7       burnt                            4.952077   
8      cheesy                            4.952077   
9   aldehydic                            3.807242   